### imports

In [ ]:
######## 미감면 이자율(금액조정 후 상환유예에서 또다시 금액조정되는 경우) 관련 문제 확인하기 위해 만든 파일. ########
import pandas as pd
import functions
from pathlib import Path
from os.path import join

In [2]:
wd = r"D:\3.자산\신용회복\솔림\1.신용회복\2.동의회신 및 채무조정안\채무조정안 및 동의회신 마감\2025년"
excel_files = Path(wd).glob("*.xlsx")
dfs = []

In [3]:
for file in excel_files:
    xls = pd.ExcelFile(file)

    if "데이터_필터가능" in xls.sheet_names:
        sheet = "데이터_필터가능"
    elif "데이터" in xls.sheet_names:
        sheet = "데이터"
    else:
        print(f"'{file.name}' 에는 '데이터_필터가능' 또는 '데이터' 시트가 없습니다.")
        continue  # 둘 다 없으면 건너뜀

    df = pd.read_excel(xls, sheet_name=sheet, skiprows=2)
    dfs.append(df)

result = pd.concat(dfs, ignore_index=True)


In [19]:
result.columns

Index(['계좌번호', '채무자키', '이름', '채권구분', '현재담당자(매각유의)', '이전담당자', '지점', '총계좌수',
       '채무구분', '이름일치여부', '주민+계좌', '이름2', '심의회차', '신청인 주민번호', '접수번호', '기관코드',
       '지로코드', '계좌번호2', '대출과목', '무보증 채권액', '무보증 약정이율', '무보증 조정이율', '담보 채권액',
       '담보 약정이율', '담보 조정이율', '보증 채권액', '보증 약정이율', '보증 조정이율', '조정전 원금',
       '조정전 이자', '조정전 연체이자', '조정전 비용', '조정후 원금', '조정후 이자', '조정후 연체이자',
       '조정후 비용', '변제기유예', '거치기간', '담보권실행유예기간', '특별면책', '상환기간', '현 적용 원금감면율',
       '재조정 이전 원금감면율', '현 적용 총감면율', '재조정 이전 총감면율', '가압류배정액', '보증인가압류배정액',
       '채무구분3', '주채무자 성명', '주채무자주민번호', '원금감면여부', '상환방식', '무담보 의결권비율',
       '타기관의결권 최대값', '원금감면율', '파일명', '채권사', '신청구분', '이행중재조정여부', '종합의견', '조정구분',
       '재조정 신청 사유', '조정중계좌수', '계좌수 확인', '과거원금감면율', '과거총감면율', '감면율조정추이',
       '요청감면율추이', '최근상세기술', '회신기록', '동의', '이행중재조정여부2', '신복이름', '전산이름', '이중채권사',
       '이행중재조정횟수'],
      dtype='object')

In [4]:
result.to_excel(Path(wd) / "통합_채무조정안_및_동의회신_2025년.xlsx", index=False)

### 당시 상환후잔액 붙이기

In [41]:
desktop_path = r"C:\Users\DATA\Desktop"
fn = "통합_채무조정안_및_동의회신_2025년.xlsx"

In [42]:
data = pd.read_excel(join(desktop_path, fn), sheet_name="data", usecols=["신용회복키", "심의회차"], dtype={"신용회복키": str, "심의회차": str})
date_info = pd.read_excel(join(desktop_path, fn), sheet_name="날짜", dtype={"심의회차": str}).rename(columns={"신복회신기한":"회신기한"})

In [54]:
# 상환후잔액log
상환후잔액log = pd.read_excel(join(desktop_path, "상환후잔액로그_20260107_0938.xlsx"), usecols=["신용회복키","변경전상환후잔액","변경후상환후잔액","변경일"], dtype={"신용회복키": str})
상환후잔액log["변경일"] = pd.to_datetime(상환후잔액log["변경일"], errors="coerce")

c:\Users\DATA\AppData\Local\anaconda3\envs\py_312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [55]:
data1 = data.merge(date_info, on="심의회차", how="left")

In [56]:
data1.columns

Index(['신용회복키', '심의회차', '통지일', '회신기한'], dtype='object')

In [59]:
data1 = data1.sort_values(["통지일","신용회복키"]) # left on, right on 이 최우선 정렬 순서임
상환후잔액log = 상환후잔액log.sort_values(["변경일", "신용회복키"])


result2 = pd.merge_asof(
    data1,
    상환후잔액log,
    by="신용회복키",
    left_on="통지일",
    right_on="변경일",
    direction="backward" # 통지일보다 이른 변경일 중 가장 가까운 값
)


In [60]:
result2["통지일"] = result2["통지일"].dt.strftime("%Y-%m-%d")
result2["회신기한"] = result2["회신기한"].dt.strftime("%Y-%m-%d")
result2.to_excel(join(desktop_path, "신용회복_동의회신종합_상환후잔액포함.xlsx"), index=False)

### 변경일 ~ 통지일 사이 입금확인
- 통지일 전에 작성된 data이므로 회신기한이 아니라 통지일까지만 살펴야

In [100]:
data = pd.read_excel(join(desktop_path, fn), sheet_name="data", usecols=["채무자키","계좌키", "채무구분","통지일","변경일"], dtype={"계좌키": str})
# 입금내역
deposit = pd.read_excel(join(desktop_path, "입금내역.xlsx"), usecols=["계좌키","입금적용일","입금합계","입금자구분"], dtype={"계좌키": str}).rename(columns={"입금자구분":"채무구분"})
deposit["입금적용일"] = pd.to_datetime(deposit["입금적용일"], errors="coerce")
deposit["채무구분"] = deposit["채무구분"].str.replace("채무자","주채무").str.replace("보증인","보증채무")

In [104]:
data = data.reset_index(drop=True)
data["_row_id"] = data.index

tmp = data.merge(
    deposit,
    on=["계좌키", "채무구분"],
    how="left"
)

cond = (
    (tmp["입금적용일"] >= tmp["변경일"]) &
    (tmp["입금적용일"] < tmp["통지일"])
)

tmp = tmp[cond]


deposit_sum = (
    tmp
    .groupby("_row_id")["입금합계"]
    .sum()
)

data["기간내입금합계"] = (
    data["_row_id"]
    .map(deposit_sum)
    .fillna(0)
)

In [105]:
data.to_excel(join(desktop_path, "신용회복_동의회신종합_입금내역포함.xlsx"), index=False)